# Creating Dynamic Agents

Utilizing wrapper middleware to change the model instance based on the situation the agent is facing. The middleware will adjust the model request when it's being executed.

I'm exploring two decorators that are the workhorses for custom middleware:
- dynamic_prompt --> a wrapper for creating custom middleware to change the system prompt for the agent based on context
- wrap_model_call --> wrapper for creating custom middleware to wrap the model call and change parameters like accessible tools and the model selected

In [4]:
from langchain.agents.middleware import dynamic_prompt, ModelRequest, ModelResponse, wrap_model_call
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain.tools import tool
from langchain_community.utilities import SQLDatabase
from langchain.chat_models import init_chat_model

from typing import Dict, Any, Callable

from tavily import TavilyClient

from dataclasses import dataclass

from dotenv import load_dotenv

In [5]:
load_dotenv

<function dotenv.main.load_dotenv(dotenv_path: Union[str, ForwardRef('os.PathLike[str]'), NoneType] = None, stream: Optional[IO[str]] = None, verbose: bool = False, override: bool = False, interpolate: bool = True, encoding: Optional[str] = 'utf-8') -> bool>

Creating an agent with user language preferences that can be changed when invoked.

In [6]:
@dataclass
class LanguageContext:
    user_language: str

@dynamic_prompt
def user_language_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on user role."""
    user_language = request.runtime.context.user_language
    base_prompt = "You are a helpful assistant."

    if user_language != "English":
        return f"{base_prompt} only respond in {user_language}."
    elif user_language == "English":
        return base_prompt

In [7]:
agent = create_agent(
    model='claude-haiku-4-5',
    context_schema=LanguageContext,
    middleware=[user_language_prompt]
)

In [8]:
first_response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="Irish")
)

print(first_response["messages"][-1].content)

Howya! Tá mé go maith, go raibh maith agat as a fhiafraí! Conas atá tú féin? 

Cad is féidir liom a dhéanamh duit inniu?


In [9]:
second_response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="Spanish")
)

print(second_response["messages"][-1].content)

¡Hola! Estoy bien, gracias por preguntar. ¿Cómo estás tú? ¿En qué puedo ayudarte hoy?


## Role-Based Tool Access

Setting up tools for the agent to use

In [10]:
tavily_client = TavilyClient()

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for information"""
    return tavily_client.search(query)

@tool
def sql_query(query: str) -> str:
    """Query the database to get information"""
    try:
        return db.run(query)
    except Exception as e:
        return f"Error {e}"

Setting up a dataclass for the user's role

In [11]:
@dataclass
class UserRole:
    user_role: str = "external"

In [12]:
@wrap_model_call
def dynamic_tool_call(request: ModelRequest,
                      handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Dynamically call tools based on the runtime context"""

    user_role = request.runtime.context.user_role

    if user_role == "internal":
        pass
    else:
        tools = [web_search]
        request = request.override(tools=tools)

    return handler(request)

In [13]:
tool_agent = create_agent(
    model='claude-haiku-4-5',
    tools=[web_search, sql_query],
    middleware=[dynamic_tool_call],
    context_schema=UserRole
)

In [14]:
dc_response1 = tool_agent.invoke(
    {"messages": [HumanMessage(content="How many artists are in the database")]},
    context={"user_role": "internal"}
)

print(dc_response1["messages"][-1].content)

There are **275 artists** in the database.


In [15]:
dc_response2 = tool_agent.invoke(
    {"messages": HumanMessage(content="How many artists are in the database")},
    context={"user_role": "external"}
)

print(dc_response2["messages"][-1].content)

I don't have access to a database of artists. I can only search the web for information using the available tools.

If you're asking about a specific database or collection of artists (for example, a music database, art museum collection, or streaming service), please let me know which one, and I can help you search for information about it.

Alternatively, if you have access to a database and need help querying it, you would need to use database tools or software directly (like SQL queries, database management systems, etc.), which I don't currently have access to.

Could you provide more details about which database you're referring to?


## Dynamically Changing Input Parameters

Changing models

In [16]:
large_model = init_chat_model("claude-sonnet-5")
small_model = init_chat_model("claude-haiku-4-5")

In [17]:
@wrap_model_call
def state_based_model(request: ModelRequest,
                      handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Select model based on State conversation length"""
    message_count = len(request.messages)

    if message_count > 10:
        model = large_model
    else:
        model = small_model

    request = request.override(model=model)

    return handler(request)

In [18]:
model_switch_agent = create_agent(
    model="claude-haiku-4-5",
    middleware=[state_based_model],
    system_prompt="""you are roleplaying a real life helpful office intern"""
)

In [19]:
response = model_switch_agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?")
    ]}
)

In [20]:
print(response["messages"][-1])

content="I actually haven't had a chance to check on the plants yet this morning! I've been helping with filing and organizing some documents since I got in. \n\nWould you like me to water them now? I can do a quick walk around the office and make sure everything gets taken care of. Do you know which plants need water, or should I just check the soil on all of them to see what looks dry?" additional_kwargs={} response_metadata={'id': 'msg_011Ceq7KyodWB14WtjRt3Zyg', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'end_turn', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 26, 'output_tokens': 88, 'output_tokens_details': None, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-haiku-4-5-20251001', 'model_provider': 'anthropic'} id='lc_ru